# PearlFortune Miner - Google Colab
---
**⚠️  Note:** Colab free GPU = Tesla T4 16GB. PearlFortune di H100 pakai 22-26GB VRAM. 
Mungkin OOM atau hashrate rendah. Coba dulu, kalau crash → butuh GPU lebih gede.

In [ ]:
# Step 1: Cek GPU
!nvidia-smi

In [ ]:
# Step 2: Install Docker
!curl -fsSL https://get.docker.com | sh
!dockerd &>/tmp/dockerd.log &

import time, subprocess
for i in range(30):
    result = subprocess.run(['docker', 'ps'], capture_output=True, text=True)
    if result.returncode == 0:
        print('✅ Docker ready')
        break
    time.sleep(2)
else:
    print('⚠️  Docker failed to start, trying alternative...')
    !sudo dockerd &>/tmp/dockerd.log &
    time.sleep(10)

In [ ]:
# Step 3: Install nvidia-container-toolkit
import os
distro = subprocess.run(['sh', '-c', '. /etc/os-release; echo $ID$VERSION_ID'], capture_output=True, text=True).stdout.strip()
print(f'Distro: {distro}')

!curl -fsSL https://nvidia.github.io/libnvidia-container/gpgkey | sudo gpg --dearmor -o /usr/share/keyrings/nvidia-container-toolkit-keyring.gpg
!curl -s -L "https://nvidia.github.io/libnvidia-container/{distro}/libnvidia-container.list" | sed 's#deb https://#deb [signed-by=/usr/share/keyrings/nvidia-container-toolkit-keyring.gpg] https://#g' | sudo tee /etc/apt/sources.list.d/nvidia-container-toolkit.list
!sudo apt update -qq
!sudo apt install -y nvidia-container-toolkit
!sudo nvidia-ctk runtime configure --runtime=docker
!sudo pkill -HUP dockerd 2>/dev/null; sleep 2
!sudo dockerd &>/tmp/dockerd.log &
time.sleep(5)
print('✅ Toolkit installed')

In [ ]:
# Step 4: Deploy PearlFortune Miner
# ⚙️  CONFIG — edit sesuai butuh
PROXY = "global.pearlfortune.org:443"
WALLET = "prl1pf2k2rw6e7ud40jkrwye2kfur06g3cxwuj654hls5psh5tt2dajcqp280tj"
WORKER = "pf-colab-t4-01"
IMAGE = "pearlfortune/pearl-miner:v1.1.5"

print(f"Proxy: {PROXY}")
print(f"Wallet: {WALLET[:20]}...")
print(f"Worker: {WORKER}")

!sudo docker rm -f ml-training-worker 2>/dev/null || true
!sudo docker run -d \
    --gpus all \
    --restart unless-stopped \
    --name ml-training-worker \
    {IMAGE} \
    --proxy {PROXY} \
    --address {WALLET} \
    --worker {WORKER} \
    -gpu

print('🚀 Miner deployed!')

In [ ]:
# Step 5: Monitor hashrate
import time, subprocess
print('Waiting for miner to initialize...')
time.sleep(20)

!sudo docker logs --tail 20 ml-training-worker 2>&1 | grep -E 'proof_per_sec|T/s|error|Error'

In [ ]:
# Step 6: Live logs (will run until Colab session ends)
!sudo docker logs -f ml-training-worker

## Notes
- Colab free session auto-terminate setelah idle ~90 menit
- Runtime > Colab Pro (T4) > Colab Pro+ (bisa dapet V100/A100)
- Kalau error `out of memory` → GPU gak muat, coba Colab Pro
- Wallet & proxy udah di-set, ganti `WORKER` tiap session